# GlacierGuards: SAR-only and optical-only ablations (50-lake independent test)
Runs ONLY the two `--split test` ablation trainings (about 30 min each on a T4). Do not add the other training cells.
Settings: Accelerator GPU T4 x1, Internet ON. Input: the `glacierguards-gloid` dataset.
Same hyper-parameters as the final fused model `seg_fused_test.pt`: 40 epochs, batch 16, lr 3e-4.

In [ ]:
import glob, os, shutil, subprocess, json
DATA = os.path.dirname(glob.glob('/kaggle/input/**/splits.json', recursive=True)[0])
shutil.copytree(f'{DATA}/code', '/kaggle/working/code', dirs_exist_ok=True)
os.chdir('/kaggle/working/code')
subprocess.run('pip -q install segmentation-models-pytorch rasterio pyproj pyyaml', shell=True)
import torch
print('DATA =', DATA)
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - turn on GPU T4 x1 in settings!')
sp = json.load(open(f'{DATA}/splits.json')); print('train_pool', len(sp['train_pool']), '| test_lakes', len(sp['test_lakes']))
COMMON = f'--data-root {DATA} --split test --epochs 40 --batch-size 16 --lr 3e-4'

In [ ]:
# 1/2: SAR-only (VV, VH) -> models/seg_sar_test.pt + seg_sar_test_metrics.json
!python src/training/train_segmentation.py {COMMON} --channels sar

In [ ]:
# 2/2: optical-only -> models/seg_optical_test.pt + seg_optical_test_metrics.json
!python src/training/train_segmentation.py {COMMON} --channels optical

In [ ]:
# check outputs and pack ONLY the 4 ablation files
import zipfile
files = ['models/seg_sar_test.pt', 'models/seg_sar_test_metrics.json', 'models/seg_optical_test.pt', 'models/seg_optical_test_metrics.json']
for f in files:
    assert os.path.exists(f), f'MISSING {f} - a training run failed, read its output above'
for n in ['seg_sar_test', 'seg_optical_test']:
    m = json.load(open(f'models/{n}_metrics.json'))
    print(f"{n}: channels={m['channels']} split={m['split']} best_epoch={m['best_epoch']} test IoU={m['iou']:.3f} F1={m['f1']:.3f}")
with zipfile.ZipFile('/kaggle/working/ablation_results.zip', 'w') as z:
    for f in files: z.write(f, os.path.basename(f))
print('ablation_results.zip', os.path.getsize('/kaggle/working/ablation_results.zip') // 2**20, 'MB  -> download it from the Output tab')